# Stage 3b: BanglaT5 Validation Inference & Comparative Scoring

This notebook runs batched beam search inference using the fine-tuned **`banglat5_final`** model over the held-out validation set (`sft_val.csv`), scores the predictions with the official competition composite metric, and performs a direct side-by-side comparison against the decoder-only baseline (`val_predictions.csv`).

### 1. Load Fine-Tuned BanglaT5 Model & Inference Utilities

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('src'))
import inference_utils
import metric_utils

model_path = "working/banglat5_final" if os.path.exists("working/banglat5_final") else "banglat5_final"

print(f"Loading BanglaT5 from: {model_path}")
model, tokenizer = inference_utils.load_banglat5_for_inference(model_path)

### 2. Run Batched Inference on Validation Split (`sft_val.csv`)

In [ ]:
val_path = "working/sft_val.csv" if os.path.exists("working/sft_val.csv") else "sft_val.csv"
val_df = pd.read_csv(val_path)
print(f"Loaded validation set: {len(val_df)} rows from {val_path}")

# Run batched beam search inference
val_pred_banglat5_df = inference_utils.run_inference_banglat5(
    model=model,
    tokenizer=tokenizer,
    df=val_df[["id", "input"]],
    batch_size=16,
    num_beams=5,
    no_repeat_ngram_size=3,
    length_penalty=0.6,
    max_new_tokens=300,
    min_new_tokens=40
)

output_val_pred_path = "working/val_predictions_banglat5.csv"
val_pred_banglat5_df.to_csv(output_val_pred_path, index=False)
print(f"Saved BanglaT5 validation predictions to {output_val_pred_path} ({len(val_pred_banglat5_df)} rows).")

### 3. Evaluate BanglaT5 Against Ground Truth

In [ ]:
print("=== Scoring BanglaT5 Validation Predictions ===")
banglat5_val_score = metric_utils.score_predictions_csv(
    output_val_pred_path,
    val_path,
    id_col="id",
    pred_col="output",
    ref_col="output"
)
print(f"\nBanglaT5 Mean Composite Score: {banglat5_val_score:.4f}")

### 4. Side-by-Side Comparison: BanglaT5 vs Decoder-Only (TituLLM-3B)

In [ ]:
decoder_val_path = "working/val_predictions.csv" if os.path.exists("working/val_predictions.csv") else "val_predictions.csv"

print("\n===========================================================")
print("       ARCHITECTURAL VALIDATION COMPARISON")
print("===========================================================")

if os.path.exists(decoder_val_path):
    print(f"Found decoder-only validation predictions at: {decoder_val_path}")
    decoder_score = metric_utils.score_predictions_csv(decoder_val_path, val_path)
    
    delta = banglat5_val_score - decoder_score
    delta_sign = "+" if delta >= 0 else ""
    
    print("\n------------------ SUMMARY COMPARISON --------------------")
    print(f"BanglaT5 (Seq2Seq, 247M params):        Composite Score = {banglat5_val_score:.4f}")
    print(f"TituLLM-3B (Decoder-Only, 3B params):   Composite Score = {decoder_score:.4f}")
    print(f"Score Delta (BanglaT5 - Decoder-Only):   {delta_sign}{delta:.4f}")
    
    if delta > 0.0001:
        print(f"\n>>> VERDICT: BanglaT5 is currently AHEAD by +{delta:.4f} points! <<<")
    elif delta < -0.0001:
        print(f"\n>>> VERDICT: BanglaT5 is currently BEHIND by {delta:.4f} points! <<<")
    else:
        print(f"\n>>> VERDICT: BanglaT5 and Decoder-Only are TIED (delta: {delta:.4f})! <<<")
else:
    print(f"Note: Decoder-only predictions ({decoder_val_path}) not yet found.")
    print(f"Run '03_inference_and_submit.ipynb' in validation mode to enable side-by-side comparison.")
    print(f"BanglaT5 Validation Score: {banglat5_val_score:.4f}")
print("===========================================================")